# BirdNET-Pi Species Stats

Local offline equivalent of the BirdNET-Pi statistics dashboard.
Connects directly to your SQLite database and recording files — no server required.

**Setup:** `pip install -r ../requirements-notebook.txt`

**Usage:** Edit the paths below, then `Kernel → Restart & Run All`.

In [ ]:
import os

# ── Edit these paths to match your setup ──────────────────────────────────────
DB_PATH        = os.path.expanduser('~/BirdNET-Pi/scripts/birds.db')
RECORDINGS_DIR = os.path.expanduser('~/BirdSongs/Extracted')
LATITUDE       = 52.0    # decimal degrees, for sunrise/sunset overlay
LONGITUDE      = 0.0
# ─────────────────────────────────────────────────────────────────────────────

if not os.path.exists(DB_PATH):
    raise FileNotFoundError(f'Database not found: {DB_PATH}')
if not os.path.isdir(RECORDINGS_DIR):
    print(f'Warning: recordings directory not found: {RECORDINGS_DIR}')
print('Config OK')

In [ ]:
import sys
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, Image, Audio, clear_output
import plotly.io as pio

sys.path.insert(0, os.path.dirname(os.path.abspath('helpers.py')))
from helpers import (
    get_connection, load_data,
    apply_date_filter, apply_resample, RESAMPLE_MAP,
    hms_to_dec, hms_to_str, sunrise_sunset_scatter,
    polar_trace, POLAR_LAYOUT,
    recording_path,
)

pio.templates.default = 'plotly_white'

conn   = get_connection(DB_PATH)
df_all = load_data(conn)
print(f'Loaded {len(df_all):,} detections across {df_all["Com_Name"].nunique()} species')
print(f'Date range: {df_all.index.min().date()} \u2192 {df_all.index.max().date()}')

In [ ]:
from datetime import timedelta

min_date = df_all.index.min().date()
max_date = df_all.index.max().date()
all_species = sorted([str(s) for s in df_all['Com_Name'].unique()])

w_start      = widgets.DatePicker(description='Start',     value=min_date, continuous_update=False)
w_end        = widgets.DatePicker(description='End',       value=max_date, continuous_update=False)
w_resample   = widgets.RadioButtons(options=list(RESAMPLE_MAP), value='15 minutes', description='Resample:')
w_topn       = widgets.IntSlider(description='Top N',      min=1, max=min(50, len(all_species)), value=10, continuous_update=False)
w_species    = widgets.Dropdown(description='Species',     options=all_species, value=all_species[0])
w_recording  = widgets.Dropdown(description='Recording',   options=[], value=None)
w_colorscale = widgets.Dropdown(description='Heatmap pal', options=px.colors.named_colorscales(), value='blues')

out_charts = widgets.Output()
out_audio  = widgets.Output()


def show_recording(recordings_df):
    filename = w_recording.value
    if filename is None or recordings_df.empty:
        return
    matches = recordings_df[recordings_df['File_Name'] == filename]
    if matches.empty:
        return
    wav, png = recording_path(matches.iloc[0], RECORDINGS_DIR)
    with out_audio:
        clear_output(wait=True)
        display(Image(filename=png, width=600)) if os.path.exists(png) else print(f'Spectrogram not found: {png}')
        display(Audio(filename=wav))             if os.path.exists(wav) else print(f'Recording not found: {wav}')


def render(change=None):
    start, end = w_start.value, w_end.value
    if not start or not end or start > end:
        return

    df            = apply_date_filter(df_all, start, end)
    if df.empty:
        with out_charts:
            clear_output()
            print('No detections in selected date range.')
        return

    df5           = apply_resample(df, w_resample.value)
    counts        = df5.value_counts()
    hourly        = pd.crosstab(df5, df5.index.hour, dropna=True, margins=True)
    top_n         = min(w_topn.value, len(counts))
    top_species   = counts[:top_n]
    selected      = str(w_species.value) if w_species.value else None

    with out_charts:
        clear_output(wait=True)

        # Top-N bar chart
        go.Figure(
            go.Bar(y=top_species.index.tolist(), x=top_species.values.tolist(),
                   orientation='h', marker_color='seagreen')
        ).update_layout(
            title=f'Top {top_n} Species  {start} \u2192 {end}',
            yaxis={'categoryorder': 'total ascending'},
            margin=dict(l=0, r=0, t=40, b=0),
            height=max(300, top_n * 25)
        ).show()

        if selected is None or selected not in hourly.index:
            return

        # Hourly polar chart
        go.Figure(polar_trace(hourly.loc[selected])).update_layout(
            title=f'{selected} \u2014 detections by hour  {start} \u2192 {end}',
            polar=POLAR_LAYOUT, height=450
        ).show()

        if w_resample.value == 'Daily':
            # Time-of-day heatmap
            sp = df['Com_Name'][df['Com_Name'] == selected].resample('15min').count()
            sp.index = [sp.index.date, sp.index.time]
            pivot = sp.unstack().fillna(0)
            fig_dec = [hms_to_dec(t) for t in pivot.columns]
            fig_str = [hms_to_str(t) for t in pivot.columns]
            step    = max(1, len(fig_str) // 12)
            pivot.columns = fig_dec
            x_ss, y_ss, t_ss = sunrise_sunset_scatter(pivot.index.tolist(), LATITUDE, LONGITUDE)
            go.Figure(data=[
                go.Heatmap(x=[d.strftime('%d-%m-%Y') for d in pivot.index],
                           y=fig_dec, z=pivot.values.T.tolist(),
                           colorscale=w_colorscale.value, showscale=False),
                go.Scatter(x=x_ss, y=y_ss, mode='lines', hoverinfo='text',
                           text=t_ss, line_color='orange', line_width=1, name=' '),
            ]).update_layout(
                title=f'{selected} \u2014 detection heatmap',
                yaxis=dict(tickmode='array', tickvals=fig_dec[::step], ticktext=fig_str[::step]),
                height=500
            ).show()
        else:
            # Daily bar chart
            daily = pd.crosstab(df5, df5.index.date, margins=True)
            if selected in daily.index:
                row = daily.loc[selected]
                sp_df = df[df['Com_Name'] == selected]
                go.Figure(
                    go.Bar(x=[str(c) for c in daily.columns[:-1]],
                           y=row.iloc[:-1].tolist(), marker_color='seagreen')
                ).update_layout(
                    title=(f'{selected} \u2014 daily detections  '
                           f'Total: {int(hourly.loc[selected, "All"]):,}  '
                           f'Max conf: {sp_df["Confidence"].max()*100:.1f}%  '
                           f'Median: {sp_df["Confidence"].median()*100:.1f}%'),
                    height=350
                ).show()

    # Recording dropdown
    recs = df[df['Com_Name'] == selected][['Date', 'Directory', 'File_Name', 'Confidence']].sort_index(ascending=False)
    rec_opts = [(f"{r.Index.date()} {r.Index.time().strftime('%H:%M')} ({r.Confidence*100:.0f}%) \u2014 {r.File_Name}",
                  r.File_Name) for r in recs.itertuples()]
    w_recording.unobserve_all()
    w_recording.options = []
    w_recording.options = rec_opts
    w_recording.observe(lambda c: show_recording(recs), names='value')
    show_recording(recs)


for w in (w_start, w_end, w_resample, w_topn, w_colorscale, w_species):
    w.observe(render, names='value')

display(
    widgets.HBox([
        widgets.VBox([w_start, w_end, w_resample, w_topn]),
        widgets.VBox([w_species, w_recording, w_colorscale]),
    ]),
    out_charts,
    out_audio,
)
render()